# NVDA backtest — Alpaca + Yahoo news + FinBERT + XGBoost

End-to-end **exploration** notebook for one symbol (NVIDIA). Mirrors the production pipeline at a high level:

| Step | Source | Role |
|------|--------|------|
| Price data | **Alpaca** | Daily OHLCV bars |
| News | **Yahoo Finance** (`yfinance`) | Headlines + publish time |
| Sentiment | **FinBERT API** | Per-article label + scores |
| Features | Python helpers | RSI, SMA distance, volume, sentiment aggregates |
| Model | **XGBoost** (trained here) | Predict 5-day forward direction |
| Baseline | Rule-based | FinBERT + RSI thresholds (similar spirit to Go `ComputeDecision`) |

**Env:** repo root `.env` — needs `ALPACA_API_KEY`, `ALPACA_API_SECRET`, `FINBERT_API_URL`, `FINBERT_API_KEY`.

**Limitation:** Yahoo news via `yfinance` is mostly **recent** (~100 articles). The backtest window is clipped to dates where we have news. For full historical replay use the Go calibration CLI on Postgres snapshots (`docs/runbooks/calibration-backtest.md`).

In [14]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

NOTEBOOKS = Path.cwd()
if NOTEBOOKS.name != "notebooks":
    NOTEBOOKS = NOTEBOOKS / "notebooks"
sys.path.insert(0, str(NOTEBOOKS))
sys.path.insert(0, str(NOTEBOOKS / "explorations"))

load_dotenv(NOTEBOOKS.parent / ".env")

from finbert_client import finbert_analyze_many, finbert_health
from alpaca_data import fetch_daily_bars
from yahoo_news import fetch_yahoo_news, news_for_trading_day
from features import (
    FEATURE_COLUMNS,
    add_forward_targets,
    add_technical_features,
    aggregate_sentiment,
)
from backtest import BacktestResult, classification_metrics, run_long_only_backtest

SYMBOL = "NVDA"
LOOKBACK_DAYS = 400
FORWARD_HORIZON = 5
TRAIN_RATIO = 0.7

print("FinBERT:", finbert_health())

ConnectionError: HTTPSConnectionPool(host='finbert.example.com', port=443): Max retries exceeded with url: /health (Caused by NameResolutionError("HTTPSConnection(host='finbert.example.com', port=443): Failed to resolve 'finbert.example.com' ([Errno 11001] getaddrinfo failed)"))

In [ ]:
SYMBOL = "NVDA"
LOOKBACK_DAYS = 400
FORWARD_HORIZON = 5
TRAIN_RATIO = 0.7

print("FinBERT:", finbert_health())

## 1. Alpaca — daily bars

In [1]:
bars = fetch_daily_bars(SYMBOL, days=LOOKBACK_DAYS)
bars = add_technical_features(bars)
bars = add_forward_targets(bars, horizons=(3, FORWARD_HORIZON))

print(f"{len(bars)} trading days: {bars.index.min().date()} → {bars.index.max().date()}")
bars.tail()

NameError: name 'fetch_daily_bars' is not defined

## 2. Yahoo Finance — news headlines

In [ ]:
news = fetch_yahoo_news(SYMBOL, max_items=200, symbol_filter=True)
print(f"{len(news)} articles")
if not news.empty:
    print(news["published_at"].min(), "→", news["published_at"].max())
news.head(10)

## 3. FinBERT — sentiment on headlines

We batch-analyze unique titles once, then map back to each trading day's 24h news window.

In [ ]:
unique_titles = news["title"].drop_duplicates().tolist() if not news.empty else []
title_to_sentiment: dict[str, dict] = {}

if unique_titles:
    analyses = finbert_analyze_many(unique_titles, use_batch=True)
    title_to_sentiment = dict(zip(unique_titles, analyses))

    sample = analyses[0]
    print("Sample:", unique_titles[0][:80], "→", sample["label"], sample["scores"])
else:
    print("No Yahoo news returned — sentiment columns will be zero.")

## 4. Build daily feature matrix

Clip to trading days that overlap Yahoo news coverage (exploration constraint).

In [ ]:
def sentiment_for_day(day: pd.Timestamp) -> dict[str, float]:
    day_news = news_for_trading_day(news, day, lookback_hours=24)
    analyses = [title_to_sentiment[t] for t in day_news["title"] if t in title_to_sentiment]
    return aggregate_sentiment(analyses)


panel = bars.copy()
if news.empty:
    for col in ["news_count_24h", "sentiment_mean", "sentiment_pos_ratio", "sentiment_neg_ratio", "sentiment_confidence"]:
        panel[col] = 0.0
else:
    news_start = news["published_at"].min().normalize()
    news_end = news["published_at"].max().normalize()
    panel = bars.loc[(bars.index >= news_start) & (bars.index <= news_end)].copy()
    sent_df = pd.DataFrame([sentiment_for_day(d) for d in panel.index], index=panel.index)
    panel = pd.concat([panel, sent_df], axis=1)

target_col = f"target_up_{FORWARD_HORIZON}d"
panel = panel.dropna(subset=FEATURE_COLUMNS + [target_col, "close"])

print(f"Panel rows: {len(panel)}")
panel[FEATURE_COLUMNS + [target_col]].describe().T

## 5. XGBoost — train / test split

Predict whether **close** is higher in `FORWARD_HORIZON` trading days. Features mix technical markers (RSI, SMA distance, volatility) with FinBERT sentiment aggregates.

> **Note:** The repo's `xgboost-service` serves a pre-trained model from disk. For exploration we train **inline** here. Once you have a stable feature set, export with `model.save_model()` and point the service at it.

In [ ]:
split_idx = int(len(panel) * TRAIN_RATIO)
train = panel.iloc[:split_idx]
test = panel.iloc[split_idx:]

X_train, y_train = train[FEATURE_COLUMNS], train[target_col].astype(int)
X_test, y_test = test[FEATURE_COLUMNS], test[target_col].astype(int)

model = XGBClassifier(
    n_estimators=120,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    random_state=42,
)
model.fit(X_train, y_train)

proba_test = pd.Series(model.predict_proba(X_test)[:, 1], index=test.index)
metrics = classification_metrics(y_test, proba_test, threshold=0.55)
try:
    metrics["auc"] = float(roc_auc_score(y_test, proba_test))
except ValueError:
    metrics["auc"] = float("nan")

print("Test metrics (threshold=0.55):", metrics)

importance = pd.Series(model.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=False)
importance

## 6. Backtest strategies

Compare three long-only curves on the **test** window:

1. **Buy & hold** — always in market
2. **XGBoost** — long when P(up) ≥ 0.55
3. **Rule-based** — long when FinBERT sentiment > 0.15 and RSI < 70 (simple proxy for the Go decision engine)

In [ ]:
prices = test["close"]

xgb_signal = (proba_test >= 0.55).astype(int)
rule_signal = ((test["sentiment_mean"] > 0.15) & (test["rsi14"] < 70)).astype(int)
bh_signal = pd.Series(1, index=test.index)

bt_xgb = run_long_only_backtest(prices, xgb_signal)
bt_rule = run_long_only_backtest(prices, rule_signal)
bt_bh = run_long_only_backtest(prices, bh_signal)

summary = pd.DataFrame(
    {
        "XGBoost": bt_xgb.metrics,
        "Rule (FinBERT+RSI)": bt_rule.metrics,
        "Buy & hold": bt_bh.metrics,
    }
).T
summary

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

for label, bt, color in [
    ("Buy & hold", bt_bh, "gray"),
    ("Rule-based", bt_rule, "tab:orange"),
    ("XGBoost", bt_xgb, "tab:blue"),
]:
    axes[0].plot(bt.equity.index, bt.equity.values, label=label, color=color)

axes[0].set_title(f"{SYMBOL} backtest — test window")
axes[0].set_ylabel("Equity ($)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(importance.index, importance.values, color="tab:green")
axes[1].set_title("XGBoost feature importance")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 7. Next steps

- **More history:** wire Postgres `symbol_feature_snapshots` + `market_candles` (see `go run ./cmd/calibrate/`) for multi-month replay with stored FinBERT aggregates.
- **Export model:** `model.save_model("models/nvda_xgb.json")` and mount in `xgboost-service` (`MODEL_PATH`).
- **Tune:** adjust `FORWARD_HORIZON`, probability threshold, or add features (MACD, beta, sector ETF returns).
- **Walk-forward:** retrain XGBoost on rolling windows instead of one static split.